# Advanced Text-to-Image Generator
This notebook uses **Hugging Face Stable Diffusion** with a **Gradio interface** to generate images from text prompts.

In [ ]:
# Install required packages
!pip install -q gradio diffusers transformers torch

In [ ]:
import gradio as gr
from diffusers import StableDiffusionPipeline
import torch

In [ ]:
# Load Stable Diffusion model
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def generate_image(prompt, style, guidance_scale, steps):
    final_prompt = f"{prompt}, {style}, highly detailed, professional lighting"
    image = pipe(
        final_prompt,
        guidance_scale=guidance_scale,
        num_inference_steps=steps
    ).images[0]
    return image

In [ ]:
# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("## Advanced Text-to-Image Generator")
    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(label="Prompt", placeholder="Describe your scene...")
            style_input = gr.Dropdown(
                label="Style",
                choices=["photorealistic", "digital art", "anime", "watercolor", "3D render", "concept art", "cinematic"],
                value="photorealistic"
            )
            guidance_input = gr.Slider(label="Guidance Scale", minimum=5, maximum=15, value=7.5, step=0.5)
            steps_input = gr.Slider(label="Inference Steps", minimum=10, maximum=100, value=50, step=5)
            generate_button = gr.Button("Generate Image")
        with gr.Column():
            output_image = gr.Image(label="Generated Image")

    generate_button.click(
        fn=generate_image,
        inputs=[prompt_input, style_input, guidance_input, steps_input],
        outputs=output_image
    )

demo.launch()